# 2 - Preprocessing

This notebook runs the consolidated preprocessing pipeline from `src/data_preprocess.py`.

It will: filter short / non-Vietnamese comments, normalize Unicode, decode teencode, optionally segment with VnCoreNLP, and decode emojis.

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is importable when running from notebooks/
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [ ]:
import pandas as pd

from src.data_preprocess import PreprocessConfig, build_resources, preprocess_df, save_processed_csv


In [ ]:
# Paths
DATA_DIR = REPO_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
OUT_DIR = DATA_DIR

# Optional (only needed if you want vncorenlp segmentation)
VNCORENLP_DIR = REPO_ROOT / 'src' / 'vncorenlp'

config = PreprocessConfig(
    len_threshold=40,
    space_threshold=10,
    require_vietnamese=True,
    enable_teencode=True,
    enable_emoji_decode=True,
    # Enable only if you have Java + vncorenlp model dir available
    enable_vncorenlp=VNCORENLP_DIR.exists(),
    vncorenlp_model_dir=str(VNCORENLP_DIR) if VNCORENLP_DIR.exists() else None,
    java_home=None,
)

resources = build_resources(config)
if config.enable_vncorenlp and not resources.vncorenlp_available:
    print('VnCoreNLP disabled:', resources.vncorenlp_error)


In [ ]:
# Process one file (change filename as needed)
example_file = next(RAW_DIR.glob('*.csv'))
print('Using:', example_file.name)
df = pd.read_csv(example_file)
df_processed = preprocess_df(df, text_col='text', config=config, resources=resources)
df_processed.head()


In [ ]:
# Save processed file
out_file = OUT_DIR / f"{example_file.stem}_processed.csv"
save_processed_csv(df_processed, str(out_file), text_col='text')
print('Saved to:', out_file)


In [ ]:
# Optional: combine all *_processed.csv in data/ into data/combined_processed.csv
processed_files = sorted(OUT_DIR.glob('*_processed.csv'))
print('Found processed:', len(processed_files))
if processed_files:
    combined = pd.concat([pd.read_csv(p) for p in processed_files], ignore_index=True)
    combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)
    combined_out = OUT_DIR / 'combined_processed.csv'
    combined.to_csv(combined_out, index=True, encoding='utf-8-sig', index_label='no')
    print('Saved combined to:', combined_out)
